# 06 `fpos` Vs `fmiss`: XAI And Target Asymmetry

This notebook confronts the two targets directly. The goal is not only to show that their winners differ, but to show why the thesis treats them as different scientific problems.

**Questions answered here**
- Which features dominate the best `fpos` model?
- How do family-level behaviors differ between the best `fpos` and best `fmiss` lines?
- Where is the strongest evidence that the two targets should not share one final recipe?


## Model Architecture Map (Compared Here, Rebuild-Level)

### Shared Mechanic (Global)

1. **Split protocol for both models**: `recording_disjoint_main` with strict recording-disjoint holdout (`paired_final_holdout_rows=100`) and no holdout recording leakage into fine-tune or unlabeled pools.

### Best `fpos` (`wf_embed_anchor_stack_xgboost`) — Full Mechanism
- Runner/family: `waveform_stack` signal-embedding stack.
- Feature path: filtered context stack columns + `source_pred` + anchor/support columns + waveform latent columns `wf_embed_*` from convolutional autoencoder.
- Source transfer path: three grouped-OOF LightGBM teachers build ensemble priors; selected teacher output becomes `source_pred`.
- Unlabeled/pseudo path: pseudo candidates from PCA(10)+KNN(16)+KMeans(4) support geometry and teacher disagreement thresholds; quotas by recording/cluster.
- Trust/manifold/anchor path: pseudo weights trust-adjusted from cluster/study reliability (`cluster_trust_power=1.35`, `study_trust_weight=0.30`, `min_trust=0.25`); anchor score from HistGradientBoostingClassifier contributes `anchor_fp_score` and `anchor_support_margin`.
- Final learner and fitting: weighted XGBoost on labeled + trust-selected pseudo rows with grouped OOF logic that excludes pseudo rows sharing validation recordings.
- SSL or representation learning: waveform autoencoder embedding yes; SSL domain-adaptation pretraining no.

### Best `fmiss` (`source_plus_reduced_latent_fullctx_lightgbm`) — Full Mechanism
- Runner/family: `context_stack` reduced-latent winner variant.
- Feature path: constrained stack frame (`context + source_pred + anchor_fp_score`) instead of full anchor/support waveform stack.
- Source transfer path: same teacher ensemble strategy; `source_pred` carries source-transfer prior.
- Unlabeled/pseudo path: same manifold pseudo-selection and quota logic as stack family variants.
- Trust/manifold/anchor path: trust reweighting remains active for pseudo rows; anchor score retained, while extra support-margin columns are intentionally dropped in final reduced frame.
- Final learner and fitting: weighted LightGBM on labeled + trust-selected pseudo rows with grouped OOF selection.
- SSL or representation learning: none (no waveform embedding, no SSL pretraining).

### Direct Architectural Difference
- `fpos` winner adds representation capacity (`wf_embed_*`) on top of the anchor stack and benefits from it.
- `fmiss` winner is explicitly capacity-constrained to avoid transfer overfitting and keeps only the reduced context-transfer stack.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent
SRC = ROOT / "src"
if not SRC.exists():
    raise FileNotFoundError(f"Expected thesis src at {SRC}; run notebooks from thesis/notebooks")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()

fig_dir, table_dir = notebook_output_dirs("06_fpos_vs_fmiss_xai")
shap_df = load_fpos_shap_importance()
shap_group_df = build_shap_group_summary()
fpos_family = build_family_summary("fpos_waveform_winner", "fpos", "recording_disjoint_main")
fmiss_family = build_family_summary("fmiss_reduced_latent", "fmiss", "recording_disjoint_main")
asymmetry = build_target_asymmetry_table()
fpos_progress = build_benchmark_progress_table("fpos")
fmiss_progress = build_benchmark_progress_table("fmiss")


## 0. Load the two best models explicitly

Before analyzing feature importance or family behavior, we pin the specific recipe results that represent the best `fpos` and best `fmiss` paths. This makes the XAI analysis traceable back to the recipe system.

In [ ]:
PROTOCOL = "recording_disjoint_main"

best_fpos  = load_or_run_experiment("fpos_waveform_winner",  "fpos",  PROTOCOL)
best_fmiss = load_or_run_experiment("fmiss_reduced_latent",  "fmiss", PROTOCOL)

def _m(result):
    row = result.metrics.iloc[0] if not result.metrics.empty else {}
    return {"r2": float(row.get("r2", float("nan"))), "mae": float(row.get("mae", float("nan")))}

best_summary = pd.DataFrame([
    {"target": "fpos",  "recipe_id": best_fpos.recipe.recipe_id,  "label": best_fpos.recipe.label,
     "model_family": best_fpos.recipe.model_family, "feature_view": best_fpos.recipe.feature_view,
     **_m(best_fpos)},
    {"target": "fmiss", "recipe_id": best_fmiss.recipe.recipe_id, "label": best_fmiss.recipe.label,
     "model_family": best_fmiss.recipe.model_family, "feature_view": best_fmiss.recipe.feature_view,
     **_m(best_fmiss)},
])
display(best_summary)
save_table(best_summary, table_dir, "best_models_summary")

## 1. What drives the best `fpos` model?

The SHAP summary is useful because it tells us whether the winner is a black box or a structured correction model. The answer from earlier analysis was: a strong transferred prior plus waveform/amplitude/ISI corrections.


In [ ]:
display(shap_df.head(20))
display(shap_group_df)
save_table(shap_df, table_dir, "fpos_shap_importance")
save_table(shap_group_df, table_dir, "fpos_shap_group_summary")
fig, _ = plot_fpos_shap_top(shap_df, top_n=15)
save_figure(fig, fig_dir, "fpos_shap_top")
display(fig)
plt.close(fig)


## 2. Family behavior side by side

The family tables below make the asymmetry concrete. The question is not only which model is better overall, but which families reward or punish each target differently.


In [ ]:
display(fpos_family)
display(fmiss_family)
display(asymmetry)
save_table(fpos_family, table_dir, "fpos_family_summary")
save_table(fmiss_family, table_dir, "fmiss_family_summary")
save_table(asymmetry, table_dir, "fpos_vs_fmiss_family_comparison")


In [ ]:
fig, _, _ = plot_group_metric(fpos_family, group_col="study_set", metric="r2", title="Best `fpos` family behavior")
save_figure(fig, fig_dir, "fpos_family_behavior")
display(fig)
plt.close(fig)


In [ ]:
fig, _, _ = plot_group_metric(fmiss_family, group_col="study_set", metric="r2", title="Best `fmiss` family behavior")
save_figure(fig, fig_dir, "fmiss_family_behavior")
display(fig)
plt.close(fig)


In [ ]:
asymmetry["r2_gap_fpos_minus_fmiss"] = asymmetry["fpos_r2"] - asymmetry["fmiss_r2"]
display(asymmetry.sort_values("r2_gap_fpos_minus_fmiss", ascending=False))
fig, _ = plot_target_asymmetry(asymmetry)
save_figure(fig, fig_dir, "target_asymmetry_combined")
display(fig)
plt.close(fig)


## 3. Benchmark-level asymmetry

This table compresses the whole thesis finding into one summary: `fpos` gained much more from the later target-aware structure than `fmiss` did.


In [ ]:
basic_fpos = fpos_progress.loc[fpos_progress["recipe_id"] == "fpos_hybrid_plus_paired"].iloc[0]
final_fpos = fpos_progress.sort_values("r2", ascending=False).iloc[0]
basic_fmiss = fmiss_progress.loc[fmiss_progress["recipe_id"] == "fmiss_hybrid_residual"].iloc[0]
final_fmiss = fmiss_progress.sort_values("r2", ascending=False).iloc[0]
target_story = pd.DataFrame([
    {"target": "fpos", "basic_transfer_r2": basic_fpos["r2"], "final_r2": final_fpos["r2"], "gain_r2": final_fpos["r2"] - basic_fpos["r2"]},
    {"target": "fmiss", "basic_transfer_r2": basic_fmiss["r2"], "final_r2": final_fmiss["r2"], "gain_r2": final_fmiss["r2"] - basic_fmiss["r2"]},
])
display(target_story)
save_table(target_story, table_dir, "target_progress_summary")


In [ ]:
display(Markdown(
    """
## Key takeaways

- The best `fpos` model is heavily shaped by transferred prior information plus waveform/amplitude corrections.
- The best `fmiss` model is more conservative and context-first.
- Family-level asymmetry is not noise; it is evidence that `fpos` and `fmiss` respond to different kinds of transfer signal.
- This notebook is the strongest support for using separate deployment models for the two targets.
"""
))
